In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import os
import json
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

In [ ]:
from google.colab import drive

In [ ]:
path=drive.mount('/content/drive')
print(path)

Mounted at /content/drive
None


In [ ]:
dataset_path='/content/drive/MyDrive/ieee-fraud-detection/datasets'
files=os.listdir(dataset_path)
print(files)

['sample_submission.csv', 'test_identity.csv', 'test_transaction.csv', 'train_identity.csv', 'train_transaction.csv', 'y_train.npy', 'y_test.npy', 'X_train_scaled.npy', 'X_test_scaled.npy', 'feature_names.json']


In [ ]:
X_train=np.load(dataset_path+'/X_train_scaled.npy')
X_test=np.load(dataset_path+'/X_test_scaled.npy')
y_train=np.load(dataset_path+'/y_train.npy')
y_test=np.load(dataset_path+'/y_test.npy')

In [ ]:
smote=SMOTE(random_state=42)
X_train_s,y_train_s=smote.fit_resample(X_train,y_train)
print(X_train_s.shape)
print(y_train_s.shape)

(911804, 432)
(911804,)


In [ ]:
del X_train, y_train
gc.collect()

44

In [ ]:
imp_feature_idxs=[422, 8, 426, 420, 14, 1, 27, 26, 344, 308, 6, 24, 423, 9, 329, 3, 149, 19, 4, 424, 421, 430, 427, 180, 7, 358, 12, 18, 178, 47, 31, 15, 46, 141, 404, 5, 362, 0, 429, 28, 120, 41, 428, 36, 147, 418, 145, 45, 333, 332, 345, 99, 22, 21, 29, 341, 11, 2, 390, 394, 30, 425, 335, 103, 406, 37, 395, 330, 25, 137, 44, 34, 98, 152, 391, 23, 13, 405, 363, 410, 331, 132, 251, 79, 153, 60, 239, 357, 86, 95, 49, 215, 62, 146, 403, 300, 419, 338, 181, 237, 337, 88, 360, 32, 176, 133, 206, 112, 199, 70, 431, 412, 202, 126, 69, 106, 365, 55, 128, 35, 401, 307, 409, 417, 63, 177, 16, 85, 305, 367, 140, 271, 113, 125, 80, 94, 370, 61, 134, 368, 350, 400, 39, 114, 104, 250, 364, 89, 97, 282, 197, 346, 174, 339, 43, 324, 343, 111, 186, 392, 50, 184, 17, 84, 349, 124, 117, 253, 396, 136, 74, 296, 261, 247, 293, 374, 309, 119, 127, 238, 226, 402, 116, 150, 151, 348, 229, 73, 105, 135, 252, 227, 220, 159, 356, 347, 148, 352, 312, 214]

In [ ]:
X_train,X_val,y_train,y_val=train_test_split(X_train_s,y_train_s,test_size=0.2,random_state=42)

In [ ]:
X_train=X_train[:,imp_feature_idxs]
X_val=X_val[:,imp_feature_idxs]
X_test=X_test[:,imp_feature_idxs]

In [ ]:
X_train.shape

(729443, 200)

In [ ]:
class multiLayerPerceptron(nn.Module):
  def __init__(self,input_size,output_size):
    super().__init__()
    self.fc1=torch.nn.Linear(input_size,256)
    self.bc1=torch.nn.BatchNorm1d(256)
    self.drop1=torch.nn.Dropout(0.3)
    self.relu=torch.nn.ReLU()

    self.fc2=torch.nn.Linear(256,128)
    self.bc2=torch.nn.BatchNorm1d(128)
    self.drop2=torch.nn.Dropout(0.3)

    self.fc3=torch.nn.Linear(128,64)
    self.bc3=torch.nn.BatchNorm1d(64)
    self.drop3=torch.nn.Dropout(0.2)

    self.fc4=torch.nn.Linear(64,output_size)


  def forward(self,X):
    X=self.drop1(self.relu(self.bc1(self.fc1(X))))
    X=self.drop2(self.relu(self.bc2(self.fc2(X))))
    X=self.drop3(self.relu(self.bc3(self.fc3(X))))
    X=self.fc4(X)
    return X

In [ ]:
criterion=torch.nn.BCEWithLogitsLoss()

In [ ]:
if torch.cuda.is_available():
  print("yes")

yes


In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t=torch.tensor(X_train,dtype=torch.float32)
X_val_t=torch.tensor(X_val,dtype=torch.float32)
X_test_t=torch.tensor(X_test,dtype=torch.float32)

y_train_t=torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
y_val_t=torch.tensor(y_val,dtype=torch.float32).unsqueeze(1)
y_test_t=torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

train_dataset=TensorDataset(X_train_t,y_train_t)
train_loader=DataLoader(train_dataset,batch_size=512,shuffle=True)

In [ ]:
model=multiLayerPerceptron(input_size=X_train.shape[1],output_size=1).to(device)

optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)

scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',patience=3,factor=0.5)

In [ ]:
best_val_auc=0
patience=5
patience_cnt=0
epochs=50
best_model_state=None

In [ ]:
for epoch in range(epochs):
  model.train()
  train_loss=0.0
  for X_batch,y_batch in train_loader:
    X_batch,y_batch=X_batch.to(device),y_batch.to(device)
    optimizer.zero_grad()
    model_pred=model.forward(X_batch)
    loss=criterion(model_pred,y_batch)
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()*X_batch.size(0)

  train_loss/=len(train_loader.dataset)

  model.eval()
  with torch.no_grad():
    val_outputs=model.forward(X_val_t.to(device))
    val_loss=criterion(val_outputs,y_val_t.to(device)).item()
    val_prob=torch.sigmoid(val_outputs).cpu().numpy()
    val_auc=roc_auc_score(y_val_t.numpy(),val_prob)

  scheduler.step(val_loss)

  if val_auc>best_val_auc:
    best_val_auc=val_auc
    patience_cnt=0
    best_model_state=model.state_dict()
  else:
    patience_cnt+=1
    if patience_cnt>patience:
      print(f"Early Stopping at epoch: {epoch+1} with val auc: {best_val_auc}")
      break

  print(f"Epoch: {epoch+1} train loss: {train_loss} val loss: {val_loss} val auc: {val_auc}")

Epoch: 1 train loss: 0.35261618304387254 val loss: 0.2549799382686615 val auc: 0.963111756931144
Epoch: 2 train loss: 0.27045155719055175 val loss: 0.19819246232509613 val auc: 0.9781965224132145
Epoch: 3 train loss: 0.2365580953741352 val loss: 0.16957874596118927 val auc: 0.9838364595499307
Epoch: 4 train loss: 0.21561756687509456 val loss: 0.15362705290317535 val auc: 0.9870094465171508
Epoch: 5 train loss: 0.20224106135905712 val loss: 0.1447083055973053 val auc: 0.9888063844733609
Epoch: 6 train loss: 0.19131569849931934 val loss: 0.12907390296459198 val auc: 0.9904645931956849
Epoch: 7 train loss: 0.18371474649753444 val loss: 0.12756118178367615 val auc: 0.9908632649747948
Epoch: 8 train loss: 0.17717902195348675 val loss: 0.11983662843704224 val auc: 0.9918056705021058
Epoch: 9 train loss: 0.17163015267587367 val loss: 0.11631327122449875 val auc: 0.992203773773007
Epoch: 10 train loss: 0.16654162779449702 val loss: 0.1121666207909584 val auc: 0.9928783294211452
Epoch: 11 train

In [ ]:
with torch.no_grad():
  test_outputs=model.forward(X_test_t.to(device))
  test_prob=torch.sigmoid(test_outputs).cpu().numpy()
  test_pred=(test_prob>0.5).astype(int)

test_roc_auc=roc_auc_score(y_test_t.numpy(),test_prob)
test_f1=f1_score(y_test_t.numpy(),test_pred)

print(f"test auc: {test_roc_auc:.4f}")
print(f"test f1: {test_f1:.4f}")

test auc: 0.9394
test f1: 0.6019


In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import mlflow
import mlflow.pytorch

In [ ]:
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"

mlflow.set_tracking_uri("file:///content/drive/MyDrive/ieee-fraud-detection/ml-flow-experiments")
mlflow.set_experiment("fraud_model_bakeoff")

<Experiment: artifact_location='file:///content/drive/MyDrive/ieee-fraud-detection/ml-flow-experiments/492684693097444956', creation_time=1784204345668, effective_trace_archival_retention=None, experiment_id='492684693097444956', last_update_time=1784204345668, lifecycle_stage='active', name='fraud_model_bakeoff', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run(run_name="neural_net"):
  mlflow.log_param("architecture","MLP 200->256->128->64->1")
  mlflow.log_param("dropout","0.3/0.3/0.2")
  mlflow.log_param("optimizer","Adam")
  mlflow.log_param("batch_size",512)
  mlflow.log_param("num_features",200)
  mlflow.log_param("learning_rate",1e-3)
  mlflow.log_param("weight_decay",1e-4)
  mlflow.log_param("epochs",50)
  mlflow.log_param("early_stopping_patience",5)

  mlflow.log_metric("best_val_auc",best_val_auc)
  mlflow.log_metric("test_auc",test_roc_auc)
  mlflow.log_metric("test_f1",test_f1)

  mlflow.pytorch.log_model(model,name="model", input_example=X_train_t[0].unsqueeze(0), serialization_format='pickle')

2026/07/17 20:19:37 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/07/17 20:19:37 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/17 20:19:52 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version la

In [ ]:
runs = mlflow.search_runs(experiment_names=["fraud_model_bakeoff"])

finished_runs = runs[runs["status"] == "FINISHED"]

if not finished_runs.empty:
    print(finished_runs[["run_id", "status", "tags.mlflow.runName", "metrics.test_auc", "metrics.test_f1"]].to_string())
else:
    print("No finished runs found in the experiment.")

                             run_id    status tags.mlflow.runName  metrics.test_auc  metrics.test_f1
0  802b22bb298f4e9e9d4cbba43f97f3bc  FINISHED          neural_net          0.939448         0.601876
3  e6997eb70fb1466ea45d29a0d021fb51  FINISHED            lightgbm          0.961212         0.723536
4  cc984fe4e9fc462ca5d8cbdcff713108  FINISHED             xgboost          0.953905         0.679606


In [ ]:
lgbm_run_id="e6997eb70fb1466ea45d29a0d021fb51"

result=mlflow.register_model(
    model_uri=f"runs:/{lgbm_run_id}/model",
    name="fraud_detector"
)
print(result)

Successfully registered model 'fraud_detector'.
2026/07/17 20:26:53 WARNING mlflow.tracking._model_registry.fluent: Run with id e6997eb70fb1466ea45d29a0d021fb51 has no artifacts at artifact path 'model', registering model based on models:/m-2f92755661934223aa967820fb3205c6 instead
Created version '1' of model 'fraud_detector'.


<ModelVersion: aliases=[], creation_timestamp=1784320013139, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1784320013139, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='cv_auc', model_id='m-2f92755661934223aa967820fb3205c6', run_id='e6997eb70fb1466ea45d29a0d021fb51', step=0, timestamp=1784232427436, value=0.9967901373942848>,
 <Metric: dataset_digest=None, dataset_name=None, key='test_auc', model_id='m-2f92755661934223aa967820fb3205c6', run_id='e6997eb70fb1466ea45d29a0d021fb51', step=0, timestamp=1784232427526, value=0.9612124625451384>,
 <Metric: dataset_digest=None, dataset_name=None, key='test_f1', model_id='m-2f92755661934223aa967820fb3205c6', run_id='e6997eb70fb1466ea45d29a0d021fb51', step=0, timestamp=1784232427641, value=0.7235363690124187>], model_id='m-2f92755661934223aa967820fb3205c6', name='fraud_detector', params={'colsample_bytree': '0.749816047538945',
 'learning_rate': '0.28570714885887566',
 'max_depth': '10',
